In [ ]:
#Install and import libraries

# Install packages if you need them
#%pip install pandas
%pip install seaborn
%pip install scipy

# Import packages
import pandas as pd
import seaborn as sns

# For getting exgaussian function parameters
from scipy.stats import exponnorm
import numpy as np

# For making quantile plots
import matplotlib.pyplot as plt

In [ ]:
# load the raw data into a dataframe
# your path will be different
# remember to add /work/ at the beginning of the string
raw = pd.read_csv('/work/Experimental-Psycholinguistics/pilot.csv')

In [ ]:
# make a list of all the columns we want to keep
cols = ['Participant Code', 'correct', 'correctResponse', 'duration', 'response', 'state', 'type', 'Version', 'word']

In [ ]:
df = raw[cols] # make a new dataframe with only the columns in cols
df = df[df['type'] != 'practice']  # throw away the practice trials
df = df[df['response'] != 'next']
df = df[df['response'] != 'space']

In [ ]:
df['correct'] = (df['response'] == df['correctResponse']).astype(int)     # make a new column with 0 or 1 for incorrect / correct
df = df[df['correct'] == 1]                                                # keep only the correct responses 
df = df.rename(columns={'duration': 'rt'})         # rename the soa and duration columns, just to look nicer

In [ ]:
dfs = df[df['Version']=='S']    #making a dataframe for the standard version
dfm = df[df['Version']=='M']    #making a dataframe for the mixed version
dfsc = dfs[dfs['type']=='C']    #making a dataframe for the cognates in the standard version
dfmc = dfm[dfm['type']=='C']    #making a dataframe for the cognates in the mixed version
dc = df[df['type']=='C']        #making a dataframe for the cognates in both versions

In [ ]:
# find average values for the conditions. 
df_grouped = df.groupby(['type', 'state', 'Version'])['rt'].mean().reset_index()
dfsc_grouped = dfsc.groupby(['type', 'state', 'Version'])['rt'].mean().reset_index()
dfmc_grouped = dfmc.groupby(['type', 'state', 'Version'])['rt'].mean().reset_index()
dc_grouped = dc.groupby(['type', 'state', 'Version'])['rt'].mean().reset_index()
dc_grouped

In [ ]:
# make barplot of the data. this code makes it for the rt of the cognates in both S and M
sns.barplot(data=dc_grouped, x='Version', y='rt', hue='Version')            # make a barplot
sns.despine()                                                           # get rid of the lines around the top and bottom - looks much nicer!

In [ ]:
sns.barplot(data=df_grouped, x='Version', y='rt', hue='state')            # make a barplot for E vs N for each version
sns.despine()

In [ ]:
sns.barplot(data=df_grouped, x='Version', y='rt', hue='type')            # make a barplot for each type in each version
sns.despine()

In [ ]:
# Plot our response time data like Balota and Yap (2011) do. This is for the standard version

# Fit an ex-gaussian function to the data (maximum liklihood estimation)
K, loc, scale = exponnorm.fit(dfsc['rt'])

# exponorm fits an exgaussian function to the data (here our reaction time data)
# It finds the best values for K, loc, and scale
# K = shape parameter. This is the ratio of the exponential tail to the gaussian spread (K = tau/sigma)
# loc = how much the distribution should shift right or left
# scale = sigma (the standard deviation of the gaussian component of the distribution)

# Recover parameters (give them the same names as in Balota & Yap, 2011)
mu    = loc                            # mu is the same as loc
sigma = scale                          # sigma is the same as scale
tau   = K * scale                      # mean of the exponential component
mean = mu + tau                        # the "theoretical" mean of the fitted function
empmean = dfsc['rt'].mean()             # the "empirical" mean of the data (what "mean" normally, uhh.. means)

# Make a figure like the ones in Balota & Yap (2011). 
ax = sns.kdeplot(data = dfsc, x = 'rt', color = 'black')                                  # plot the data as a distribution
# uncomment the next two lines to check the fit of the function to the data
#x = np.linspace(df['rt'].min(), df['rt'].max(), 200)                                   # datapoint for plotting theoretical exgaussian 
#ax.plot(x, exponnorm.pdf(x, K, loc, scale), color='red', linestyle='--')               # plot theoretical exgauss function on top of acutal data (to check fit)
values = f"Mean = {mean:.3f}\nMu = {mu:.3f}\nSigma = {sigma:.3f}\nTau = {tau:.3f}"      # assemble the text for the legend
ax.text(0.95, 0.95, values,                                                             # x and y coordinates for annotations
        transform=ax.transAxes,                                                         # fixes text box to the figure, not the data
        ha='right', va='top',ma='left',                                                 # position and algin the text in the box
        fontsize=10)                                                                    # I mean, yeah. It's the font size
        #bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))                     # make a box around the annotation, if you want
ax.set(xlabel='Time (ms)', ylabel='Probability Density')                                # Change the axis labels
ax.tick_params(left=False, labelleft=False)                                             # Remove the ticks and values on the y-axis
ax.set_xlim(left=0)                                                                     # Start the x-axis at zero
sns.despine() 

In [ ]:
# Plot our response time data like Balota and Yap (2011) do. This is for the mixed version

# Fit an ex-gaussian function to the data (maximum liklihood estimation)
K, loc, scale = exponnorm.fit(dfmc['rt'])

# exponorm fits an exgaussian function to the data (here our reaction time data)
# It finds the best values for K, loc, and scale
# K = shape parameter. This is the ratio of the exponential tail to the gaussian spread (K = tau/sigma)
# loc = how much the distribution should shift right or left
# scale = sigma (the standard deviation of the gaussian component of the distribution)

# Recover parameters (give them the same names as in Balota & Yap, 2011)
mu    = loc                            # mu is the same as loc
sigma = scale                          # sigma is the same as scale
tau   = K * scale                      # mean of the exponential component
mean = mu + tau                        # the "theoretical" mean of the fitted function
empmean = dfmc['rt'].mean()             # the "empirical" mean of the data (what "mean" normally, uhh.. means)

# Make a figure like the ones in Balota & Yap (2011). 
ax = sns.kdeplot(data = dfmc, x = 'rt', color = 'black')                                  # plot the data as a distribution
# uncomment the next two lines to check the fit of the function to the data
#x = np.linspace(df['rt'].min(), df['rt'].max(), 200)                                   # datapoint for plotting theoretical exgaussian 
#ax.plot(x, exponnorm.pdf(x, K, loc, scale), color='red', linestyle='--')               # plot theoretical exgauss function on top of acutal data (to check fit)
values = f"Mean = {mean:.3f}\nMu = {mu:.3f}\nSigma = {sigma:.3f}\nTau = {tau:.3f}"      # assemble the text for the legend
ax.text(0.95, 0.95, values,                                                             # x and y coordinates for annotations
        transform=ax.transAxes,                                                         # fixes text box to the figure, not the data
        ha='right', va='top',ma='left',                                                 # position and algin the text in the box
        fontsize=10)                                                                    # I mean, yeah. It's the font size
        #bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))                     # make a box around the annotation, if you want
ax.set(xlabel='Time (ms)', ylabel='Probability Density')                                # Change the axis labels
ax.tick_params(left=False, labelleft=False)                                             # Remove the ticks and values on the y-axis
ax.set_xlim(left=0)                                                                     # Start the x-axis at zero
sns.despine() 

In [ ]:
quantiles = np.arange(.1, 1.0, .1)  # .1 to .9

# per-subject, per-condition quantiles
df_q = (
    df.groupby(['participant Code', 'Version'])['rt']
      .apply(lambda x: x.quantile(quantiles))
      .rename_axis(['participant', 'Version', 'quantile'])
      .reset_index(name='rt_q')
)
df_q # inspect the quantiles

In [ ]:

ax = sns.lineplot(data=df_q, x='quantile', y='rt_q', hue='Version', marker='o')
ax.set(xlabel='Quantiles', ylabel='RT (ms)')

ax.set_title('Main effect of congruence')
ax.set(xlabel='Quantiles', ylabel='RT (ms)')
ax.legend(title='Version')
sns.despine()